### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [21]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Read all PDF's from directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all PDF files recursively

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #add source info to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata["source_path"] = str(pdf_file)
                doc.metadata['file_type'] = 'pdf'     

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all the documents in the PDF directory
all_pdf_documents = process_all_pdfs("../data")


In [23]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [ ]:
chunks = split_documents(all_pdf_documents)
chunks

In [25]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        if not self.model:
            raise ValueError("Model not found")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
#Initialize embedding manager

embedding_manager = EmbeddingManager();
embedding_manager

### Vector Store

In [ ]:
class VectorStore:
    """Manages document embeddings in a chromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the chromaDB collection
            persist_directory: Directory to persist vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize chromaDB client and collection"""
        try:
            #Create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"document": "PDF document embeddingsfor RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initilizing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add document and their embeddings to vector store
        Args:
            documents: List of LangChain documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents should be equal to number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embeddings
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to collection in vector store {e}")
            raise

vectorstore = VectorStore()
vectorstore


In [ ]:
### Convert text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### Store in the Vector Store now
vectorstore.add_documents(chunks, embeddings)

In [29]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
            Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args :
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            print(results)

            #Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas' ] [0]
                distances = results['distances' ] [0]
                ids = results['ids'] [0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    #if similarity_score >= score_threshold:
                    if True:
                        retrieved_docs. append ({
                        'id': doc_id,
                        'content': document,
                        'metadata' : metadata,
                        'similarity_score': similarity_score,
                        'distance': distance,
                        'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print(f"No documents found")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


Integration of vectorDB context retrieval pipeline with augmented LLM output

In [30]:
### Simple RAG pipeline with Grow LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

## Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context="\n\n" .join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevent context found to answer the question"
    
    ### Generate the answer using Groq LLM
    prompt = """Use the following context to answer the questions concisely.
    Context: {context}
    Question: {query}
    Answer: 
    """

    response = llm.invoke(prompt.format(context=context, query=query))
    return response.content

In [ ]:
answer = rag_simple(query="Who is Aman", retriever=rag_retriever, llm=llm)
print(answer)

Retrieving documents for query: 'Wh0  Aman'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.88it/s]

Generated embeddings with shape: (1, 384)
{'ids': [['doc_1b7ec41c_4', 'doc_260b1946_4', 'doc_d4049525_4']], 'embeddings': None, 'documents': [['Languages: Java (Solved 250+ Leetcode Problems), JavaScript, C, Python\nTools : GitHub, AWS, Docker, Cursor, Claude, Codex, Vercel, Render, Figma, Canva \nL I N K E D I N\n|\nP O R T F O L I O\nR e a c t . j s ,  N o d e . j s ,  E x p r e s s . j s ,  M o n g o D B ,  J W T ,  G e m i n i  F l a s h - 1 . 5 ,  R E S T  A P I s\nR e a c t . j s ,  N o d e . j s ,  E x p r e s s . j s ,  M o n g o D B ,  J W T ,  R E S T  A P I s\nN o d e . j s ,  E x p r e s s . j s ,  S o c k e t . I O ,  M o n g o D B ,  R e d i s ,  J W T', 'Languages: Java (Solved 250+ Leetcode Problems), JavaScript, C, Python\nTools : GitHub, AWS, Docker, Cursor, Claude, Codex, Vercel, Render, Figma, Canva \nL I N K E D I N\n|\nP O R T F O L I O\nR e a c t . j s ,  N o d e . j s ,  E x p r e s s . j s ,  M o n g o D B ,  J W T ,  G e m i n i  F l a s h - 1 . 5 ,  R E S T  

Aman is a full‑stack software developer. He is proficient in Java (with 250+ solved LeetCode problems), JavaScript, C and Python, and works with modern web technologies such as React JS, Node JS, Express JS, MongoDB, JWT, REST APIs, Socket.IO and Redis. He also uses a range of development tools and platforms—including GitHub, AWS, Docker, Vercel, Render, Figma, Canva and AI assistants like Claude and Codex.
